In [1]:
!apt-get -qq update
!apt-get -qq install -y fluidsynth
%pip install -q pretty_midi

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 120314 files and directories currently installed.)
Preparing to unpack .../00-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../01-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (5.15.3+dfsg-2ubuntu0.2) ...
Selecting previously unselected package libevdev2:amd64.
Preparing to unpack .../02-libevdev2_1.12.1+dfsg-1_amd64.deb ...
Unpacking libevdev2:amd64 (1.12.1+dfsg-1) ...
Selecting previously unselected package libmtdev1:amd64.
Preparing to unpack .../03-libmtdev1_1.1.6-1build4_amd64.deb ...
Unpacking libmtdev1:

# Path setups

In [2]:
import os
import random
import subprocess

import numpy as np
import pandas as pd
import pretty_midi
from IPython.display import Audio, display

DATASET_PATH = "/kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0"
if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(
        f"DATASET_PATH does not exist: {DATASET_PATH}. Update this path."
    )

OUTPUT_DIR = "evaluation_outputs"
BASELINE_DIR = os.path.join(OUTPUT_DIR, "baselines")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BASELINE_DIR, exist_ok=True)

In [3]:
TASK1_DIR = "/kaggle/input/datasets/nafiurrahmanafnan/generated-midis/generated_midi_ae"
TASK2_DIR = "/kaggle/input/datasets/nafiurrahmanafnan/generated-midis/generated_midi_vae"
TASK3_DIR = "/kaggle/input/datasets/nafiurrahmanafnan/generated-midis/generated_midi_transformer"

def list_midi_files(dir_path, max_files=None):
    if not os.path.isdir(dir_path):
        return []
    files = [
        os.path.join(dir_path, f)
        for f in os.listdir(dir_path)
        if f.lower().endswith(".mid")
    ]
    files = sorted(files)
    if max_files is not None:
        files = files[:max_files]
    return files

task1_files = list_midi_files(TASK1_DIR, max_files=5)
task2_files = list_midi_files(TASK2_DIR, max_files=8)
task3_files = list_midi_files(TASK3_DIR, max_files=10)

print("Task 1 files:", len(task1_files))
print("Task 2 files:", len(task2_files))
print("Task 3 files:", len(task3_files))

Task 1 files: 5
Task 2 files: 8
Task 3 files: 10


In [4]:
FS = 16
N_FEATURES = 88
LOW_PITCH = 21
HIGH_PITCH = 108
VOCAB_SIZE = N_FEATURES + 1
TOKEN_THRESHOLD = 0.5
BASELINE_TOKENS = 512

In [5]:
def midi_to_piano_roll(midi_path, fs=FS):
    try:
        midi_obj = pretty_midi.PrettyMIDI(midi_path)
        roll = midi_obj.get_piano_roll(fs=fs)
        roll = roll[LOW_PITCH:HIGH_PITCH + 1, :]
        roll = (roll > 0).astype(np.float32).T
        return roll
    except Exception:
        return np.zeros((0, N_FEATURES), dtype=np.float32)


def roll_to_token_sequence(roll, threshold=TOKEN_THRESHOLD):
    tokens = []
    for t in range(roll.shape[0]):
        active = np.where(roll[t] >= threshold)[0]
        if len(active) == 0:
            tokens.append(0)
        else:
            tokens.append(int(active.max()) + 1)
    return np.array(tokens, dtype=np.int32)


def tokens_to_pretty_midi(tokens, fs=FS):
    seconds_per_step = 1.0 / fs
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0, name="baseline_piano")

    t = 0
    while t < len(tokens):
        tok = int(tokens[t])
        if tok == 0:
            t += 1
            continue

        pitch = tok - 1 + LOW_PITCH
        start = t
        t += 1
        while t < len(tokens) and int(tokens[t]) == tok:
            t += 1
        end = t

        piano.notes.append(
            pretty_midi.Note(
                velocity=90,
                pitch=int(np.clip(pitch, 0, 127)),
                start=start * seconds_per_step,
                end=end * seconds_per_step,
            )
        )

    pm.instruments.append(piano)
    return pm

In [6]:
top_files = os.listdir(DATASET_PATH)
csv_path = None
for f in top_files:
    if f.endswith(".csv"):
        csv_path = os.path.join(DATASET_PATH, f)
        break

if csv_path is None:
    raise FileNotFoundError("No metadata CSV file found in DATASET_PATH.")

df = pd.read_csv(csv_path)
df["abs_midi_path"] = df["midi_filename"].apply(
    lambda p: os.path.normpath(os.path.join(DATASET_PATH, p))
 )

if "split" in df.columns:
    train_df = df[df["split"] == "train"]
else:
    train_df = df

selected_midi_paths = [
    p for p in train_df["abs_midi_path"].head(5).tolist() if os.path.exists(p)
 ]

if not selected_midi_paths:
    raise FileNotFoundError("No MIDI files found for evaluation.")

reference_tokens = []
for midi_path in selected_midi_paths:
    roll = midi_to_piano_roll(midi_path, fs=FS)
    if roll.shape[0] == 0:
        continue
    tokens = roll_to_token_sequence(roll, threshold=TOKEN_THRESHOLD)
    reference_tokens.extend(tokens.tolist())

if not reference_tokens:
    raise ValueError("No reference tokens extracted from MIDI files.")

reference_tokens = np.array(reference_tokens, dtype=np.int32)
token_values, token_counts = np.unique(reference_tokens, return_counts=True)
token_probs = token_counts / token_counts.sum()

print("Reference MIDI files:", len(selected_midi_paths))
print("Reference tokens:", reference_tokens.shape)

Reference MIDI files: 5
Reference tokens: (53953,)


In [7]:
def sample_random_tokens(n_tokens):
    return np.random.choice(token_values, size=n_tokens, p=token_probs).astype(np.int32)

In [8]:
SOUNDFONT_DIR = "/kaggle/working/soundfonts"
os.makedirs(SOUNDFONT_DIR, exist_ok=True)
LOCAL_SF2 = os.path.join(SOUNDFONT_DIR, "FluidR3_GM.sf2")

if not os.path.exists(LOCAL_SF2):
    try:
        import urllib.request
        url = "https://github.com/FluidSynth/fluidsynth/wiki/_pages/FluidR3_GM.sf2"
        urllib.request.urlretrieve(url, LOCAL_SF2)
    except Exception as e:
        print("Could not download SoundFont:", e)

SOUNDFONT_PATHS = [
    LOCAL_SF2,
    "/usr/share/sounds/sf2/FluidR3_GM.sf2",
    "/usr/share/sounds/sf2/TimGM6mb.sf2",
]

SOUNDFONT = None
for path in SOUNDFONT_PATHS:
    if os.path.exists(path):
        SOUNDFONT = path
        break

print("SoundFont:", SOUNDFONT)

def midi_to_wav(midi_file, wav_file):
    if SOUNDFONT is None:
        return False
    command = [
        "fluidsynth", "-ni", SOUNDFONT, midi_file,
        "-F", wav_file, "-r", "44100"
    ]
    try:
        subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True
    except Exception as e:
        print("FluidSynth failed:", e)
        return False

def listen_midi(midi_file):
    wav_file = midi_file.replace(".mid", ".wav")
    made_wav = midi_to_wav(midi_file, wav_file)

    if made_wav and os.path.exists(wav_file):
        display(Audio(wav_file))
    else:
        print("Using simple pretty_midi preview. This may sound electronic.")
        music = pretty_midi.PrettyMIDI(midi_file)
        audio = music.synthesize(fs=44100)
        if np.max(np.abs(audio)) > 0:
            audio = audio / np.max(np.abs(audio))
        display(Audio(audio, rate=44100))

SoundFont: /kaggle/working/soundfonts/FluidR3_GM.sf2


# Baseline models

## Baseline 1: Random note generator

In [9]:
def create_random_midi(output_file, token_count=BASELINE_TOKENS):
    tokens = sample_random_tokens(token_count)
    pm = tokens_to_pretty_midi(tokens, fs=FS)
    pm.write(output_file)

random_baseline_file = os.path.join(BASELINE_DIR, "random_baseline.mid")
create_random_midi(random_baseline_file)
print("Saved:", random_baseline_file)

Saved: evaluation_outputs/baselines/random_baseline.mid


## Baseline 2: Markov chain generator

In [10]:
transition = np.ones((VOCAB_SIZE, VOCAB_SIZE), dtype=np.float32)

for i in range(len(reference_tokens) - 1):
    current_token = int(reference_tokens[i])
    next_token = int(reference_tokens[i + 1])
    transition[current_token, next_token] = transition[current_token, next_token] + 1

for i in range(VOCAB_SIZE):
    transition[i] = transition[i] / transition[i].sum()


def create_markov_midi(output_file, token_count=BASELINE_TOKENS):
    tokens = []
    current_token = int(np.random.choice(reference_tokens))

    for _ in range(token_count):
        next_token = int(np.random.choice(np.arange(VOCAB_SIZE), p=transition[current_token]))
        tokens.append(next_token)
        current_token = next_token

    pm = tokens_to_pretty_midi(tokens, fs=FS)
    pm.write(output_file)

markov_baseline_file = os.path.join(BASELINE_DIR, "markov_baseline.mid")
create_markov_midi(markov_baseline_file)
print("Saved:", markov_baseline_file)

Saved: evaluation_outputs/baselines/markov_baseline.mid


# Evaluation metrics

We calculate:

- Pitch histogram similarity
- Rhythm diversity
- Repetition ratio
- Average step/pause
- Average duration
- Note count

## Evaluation functions

In [11]:
def get_notes_from_midi(midi_file):
    midi = pretty_midi.PrettyMIDI(midi_file)
    notes = []

    for instrument in midi.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            notes.append(note)

    notes = sorted(notes, key=lambda n: (n.start, n.pitch))
    return notes


def pitch_histogram_from_notes(notes):
    hist = np.zeros(12, dtype=np.float32)

    for note in notes:
        hist[note.pitch % 12] = hist[note.pitch % 12] + 1

    if hist.sum() > 0:
        hist = hist / hist.sum()

    return hist


def pitch_histogram_similarity(real_midi, generated_midi):
    real_notes = get_notes_from_midi(real_midi)
    generated_notes = get_notes_from_midi(generated_midi)

    real_hist = pitch_histogram_from_notes(real_notes)
    generated_hist = pitch_histogram_from_notes(generated_notes)

    score = np.sum(np.abs(real_hist - generated_hist))
    return float(score)


def rhythm_diversity(midi_file):
    notes = get_notes_from_midi(midi_file)
    durations = []

    for note in notes:
        durations.append(round(note.end - note.start, 2))

    if len(durations) == 0:
        return 0.0

    return float(len(set(durations)) / len(durations))


def repetition_ratio(midi_file):
    notes = get_notes_from_midi(midi_file)
    pitches = [note.pitch for note in notes]

    if len(pitches) < 4:
        return 0.0

    patterns = []
    for i in range(len(pitches) - 3):
        patterns.append((pitches[i], pitches[i + 1], pitches[i + 2], pitches[i + 3]))

    repeated = 0
    for pattern in set(patterns):
        if patterns.count(pattern) > 1:
            repeated = repeated + 1

    return float(repeated / max(1, len(patterns)))


def evaluate_midi_file(reference_midi, midi_file, model_name):
    notes = get_notes_from_midi(midi_file)

    durations = []
    starts = []
    for note in notes:
        durations.append(note.end - note.start)
        starts.append(note.start)

    starts = sorted(starts)
    steps = []
    for i in range(1, len(starts)):
        steps.append(starts[i] - starts[i - 1])

    row = {
        "model": model_name,
        "file": midi_file,
        "note_count": len(notes),
        "duration_seconds": pretty_midi.PrettyMIDI(midi_file).get_end_time(),
        "pitch_histogram_distance_lower_is_better": pitch_histogram_similarity(reference_midi, midi_file),
        "rhythm_diversity": rhythm_diversity(midi_file),
        "repetition_ratio_lower_is_better": repetition_ratio(midi_file),
        "average_step_seconds": float(np.mean(steps)) if len(steps) > 0 else 0.0,
        "average_note_duration_seconds": float(np.mean(durations)) if len(durations) > 0 else 0.0,
    }

    return row

## Evaluate all outputs

In [12]:
reference_midi = selected_midi_paths[0]

baseline_random_path = "/kaggle/working/evaluation_outputs/baselines/random_baseline.mid"
baseline_markov_path = "/kaggle/working/evaluation_outputs/baselines/markov_baseline.mid"

random_path = baseline_random_path if os.path.exists(baseline_random_path) else random_baseline_file
markov_path = baseline_markov_path if os.path.exists(baseline_markov_path) else markov_baseline_file

files_to_evaluate = []

for file in task1_files:
    files_to_evaluate.append(("Task 1 LSTM Autoencoder", file))

for file in task2_files:
    files_to_evaluate.append(("Task 2 VAE", file))

for file in task3_files:
    files_to_evaluate.append(("Task 3 Transformer", file))

files_to_evaluate.append(("Random Baseline", random_path))
files_to_evaluate.append(("Markov Baseline", markov_path))

result_rows = []

for model_name, midi_file in files_to_evaluate:
    try:
        row = evaluate_midi_file(reference_midi, midi_file, model_name)
        result_rows.append(row)
    except Exception as e:
        print("Could not evaluate:", midi_file, e)

results_df = pd.DataFrame(result_rows)
results_csv = os.path.join(OUTPUT_DIR, "evaluation_metrics.csv")
results_df.to_csv(results_csv, index=False)

print("Saved metrics:", results_csv)
results_df

Saved metrics: evaluation_outputs/evaluation_metrics.csv


,model,file,note_count,duration_seconds,pitch_histogram_distance_lower_is_better,rhythm_diversity,repetition_ratio_lower_is_better,average_step_seconds,average_note_duration_seconds
0,Task 1 LSTM Autoencoder,/kaggle/input/datasets/nafiurrahmanafnan/gener...,26,8.0,0.984788,0.769231,0.000000,0.190000,3.074476
1,Task 1 LSTM Autoencoder,/kaggle/input/datasets/nafiurrahmanafnan/gener...,34,8.0,0.683570,0.852941,0.000000,0.227273,2.343717
2,Task 1 LSTM Autoencoder,/kaggle/input/datasets/nafiurrahmanafnan/gener...,15,8.0,0.922468,0.600000,0.000000,0.258929,5.320758
3,Task 1 LSTM Autoencoder,/kaggle/input/datasets/nafiurrahmanafnan/gener...,21,8.0,0.669390,0.761905,0.000000,0.381250,3.800433
4,Task 1 LSTM Autoencoder,/kaggle/input/datasets/nafiurrahmanafnan/gener...,36,8.0,0.655111,0.694444,0.000000,0.187532,2.215215
5,Task 2 VAE,/kaggle/input/datasets/nafiurrahmanafnan/gener...,39,8.0,0.786386,0.794872,0.000000,0.207237,2.046387
6,Task 2 VAE,/kaggle/input/datasets/nafiurrahmanafnan/gener...,28,8.0,0.458457,0.892857,0.000000,0.236111,2.854951
7,Task 2 VAE,/kaggle/input/datasets/nafiurrahmanafnan/gener...,49,8.0,0.637365,0.591837,0.000000,0.162784,1.628757
8,Task 2 VAE,/kaggle/input/datasets/nafiurrahmanafnan/gener...,50,8.0,0.610245,0.600000,0.000000,0.154360,1.592409
9,Task 2 VAE,/kaggle/input/datasets/nafiurrahmanafnan/gener...,38,8.0,0.567059,0.815789,0.000000,0.197666,2.095275
